# FOSI Heat Budget for The Blob

## 0. Set up

In [1]:
import cmocean
import numpy as np
import xarray as xr
import pandas as pd
import cftime
import dask
import matplotlib.pyplot as plt
import os
import xesmf as xe
import pop_tools
import json
import re
from tqdm import tqdm

import bottleneck

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter
from cartopy.util import add_cyclic_point
from matplotlib.colors import Normalize
from matplotlib.colors import LogNorm  # <-- Add this import
from matplotlib.colors import Normalize, LinearSegmentedColormap
import matplotlib
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import matplotlib.dates as mdates
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.animation import FFMpegWriter as fw
from matplotlib.animation import PillowWriter as pw

from tqdm import tqdm
from scipy.interpolate import interp1d
from scipy.stats import gaussian_kde
import imageio

from IPython.utils import io
from os.path import exists

import dask
import dask_jobqueue
import distributed
import dask.array as da

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

#### Functions

In [2]:
def pop_find_lat_ind(loc, LATDAT): # from Anna's notebook
    return np.abs(LATDAT[:, 0].values - loc).argmin()

def pop_find_lon_ind(loc, LONDAT, direction="w"): # from Anna's notebook
    if direction.lower() in ["east", "e"]:
        value = loc
    elif direction.lower() in ["west", "w"]:
        value = 360 - loc
    else:
        print("I do not know which direction.")
    return np.nanargmin(np.abs(LONDAT[0, :].values - value))

In [3]:
# Accessing glade CESM LENS2
def get_var_paths(directory, var):
    # Prefixes to match for future and historical datasets
    prefixes_to_match_fut = ['b.e21.BSSP370cmip6.', 'b.e21.BSSP370smbb.']
    prefixes_to_match_hist = ['b.e21.BHISTcmip6.', 'b.e21.BHISTsmbb.']
        
    # Sets to store unique prefixes for future and historical filenames
    prefixes_fut = list()
    prefixes_hist = list()
        
    # Iterate through files in the directory
    for filename in os.listdir(directory):
        # Check and add prefixes for future scenario files
        if any(filename.startswith(prefix) for prefix in prefixes_to_match_fut) and filename.endswith('.nc'):
            prefixes_fut.append(filename.rsplit('.', 3)[0])
            
        # Check and add prefixes for historical scenario files
        if any(filename.startswith(prefix) for prefix in prefixes_to_match_hist) and filename.endswith('.nc'):
            prefixes_hist.append(filename.rsplit('.', 3)[0])
        
    prefixes_hist_set = set(prefixes_hist)
    sorted_unique_list_hist = sorted(prefixes_hist_set)
    
    prefixes_fut_set = set(prefixes_fut)
    sorted_unique_list_fut = sorted(prefixes_fut_set)
    
    path_intermed_fut = sorted_unique_list_fut
    path_intermed_hist = sorted_unique_list_hist

    return path_intermed_hist, path_intermed_fut

def get_ds_var(directory, var, comp, index_hist):
    path_intermed_hist, path_intermed_fut = get_var_paths(directory, var)
    filename_identifier = '.'.join(path_intermed_hist[index_hist].rsplit('.', 5)[1:4])
    index_fut = find_identifier_with_index(path_intermed_hist, filename_identifier)[0][1]
    hist_file_paths = get_hist_file_paths(var,directory, path_intermed_hist, index_hist)
    fut_file_paths = get_fut_file_paths(var, directory, path_intermed_fut, index_fut)
    ds_var_fut = file_path_to_var_ds(fut_file_paths)
    ds_var_hist = file_path_to_var_ds(hist_file_paths)
    return ds_var_hist, ds_var_fut

def find_identifier_with_index(prefixes, identifier):
    """
    Find prefixes that contain a specific identifier and their indices.

    Parameters:
        prefixes (list): A list of prefixes to search through.
        identifier (str): The identifier to search for in the prefixes.

    Returns:
        list: A list of tuples containing matching prefixes and their indices.
    """
    matching_prefixes_with_indices = []  # Initialize a list to store matches and their indices

    for index, prefix in enumerate(prefixes):  # Use enumerate to get both index and prefix
        if identifier in prefix:  # Check if the identifier is in the prefix
            matching_prefixes_with_indices.append((prefix, index))  # Add the prefix and index as a tuple

    return matching_prefixes_with_indices  # Return the list of matching prefixes and indices
    
def get_hist_file_paths(var, directory, path_intermed_hist, index):
    attrib_title = path_intermed_hist[index]
    file_paths = []
    for start_year in range(1850, 2010, 10):
        end_year = start_year + 9
        file_path = f'{directory}{attrib_title}.{var}.{start_year}01-{end_year}12.nc'
        file_paths.append(file_path)
    last_file_path = f'{directory}{attrib_title}.{var}.201001-201412.nc'
    file_paths.append(last_file_path)
    return file_paths

def get_fut_file_paths(var, directory, path_intermed_fut, index):
    attrib_title = path_intermed_fut[index]
    file_paths = []
    for start_year in range(2015, 2095, 10):
        end_year = start_year + 9
        file_path = f'{directory}{attrib_title}.{var}.{start_year}01-{end_year}12.nc'
        file_paths.append(file_path)
    last_file_path = f'{directory}{attrib_title}.{var}.209501-210012.nc'
    file_paths.append(last_file_path)
    return file_paths

def file_path_to_var_ds(file_paths):
    var_ds = xr.open_mfdataset(file_paths, 
                                 concat_dim='time', 
                                 combine='nested', 
                                 parallel=True)
    return var_ds

In [4]:
def setup_axes(ax):
    ax.add_feature(cfeature.LAND, facecolor='white', zorder=2)
    ax.coastlines(resolution='110m', color='black', lw=2)
    ax.set_ylabel('latitude')
    ax.set_xlabel('longitude')


def lons_to_360(data, coord='lon'):
    """ Converts longitude coordinates from (-180, 180) to (0, 360)."""
    data.coords[coord] = (360 + (data.coords[coord] % 360)) % 360
    data = data.sortby(data[coord])
    return data


def remove_trend(da, dim, deg=1):
    # detrend along a single dimension
    # return polyfit coefficients and detrended da
    p = da.polyfit(dim=dim, deg=deg, skipna=True)
    coord = da.coords[dim]
    fit = xr.polyval(coord, p.polyfit_coefficients)
    return da - fit


def get_anoms(da):
    clim = da.groupby('time.month').mean('time')
    da_noclim = da.groupby('time.month') - clim
    anoms = remove_trend(da_noclim, dim='time')
    return anoms


def regrid_SMYLE(ds, glat=1, glon=1):
    """
    Inputs:
        ds: xr.DataArray with coordinates that include TLAT and TLONG
    Returns:
        Regridded xr.DataArray with coordinates lat and lon
    """
    ds = ds.rename(({'TLONG': 'lon', 'TLAT': 'lat'}))
    ds_out = xe.util.grid_global(glon, glat)
    regridder = xe.Regridder(ds, ds_out, 'bilinear', periodic=True)
    regridded = regridder(ds)
    new_coords = regridded.assign_coords({'y': regridded.lat[:, 0].values, 'x': regridded.lon[0].values})
    return new_coords.drop_vars(['lat', 'lon']).rename({'x': 'lon', 'y': 'lat'})

In [5]:
def calculate_anomalies_trend_features(ds):
    sst = ds
    dyr = ds.time.dt.year + ds.time.dt.month/12
    # Our 6 coefficient model is composed of the mean, trend, annual sine and cosine harmonics, & semi-annual sine and cosine harmonics
    model = np.array([np.ones(len(dyr))] + [dyr-np.mean(dyr)] + [np.sin(2*np.pi*dyr)] + [np.cos(2*np.pi*dyr)] + [np.sin(4*np.pi*dyr)] + [np.cos(4*np.pi*dyr)])
    
    # Take the pseudo-inverse of model to 'solve' least-squares problem
    pmodel = np.linalg.pinv(model)
    
    # Convert model and pmodel to xaray DataArray
    model_da = xr.DataArray(model.T, dims=['time','coeff'], coords={'time':sst.time.values, 'coeff':np.arange(1,7,1)}) 
    pmodel_da = xr.DataArray(pmodel.T, dims=['coeff','time'], coords={'coeff':np.arange(1,7,1), 'time':sst.time.values})  
    
    # resulting coefficients of the model
    sst_mod = xr.DataArray(pmodel_da.dot(sst), dims=['coeff','lat','lon'], coords={'coeff':np.arange(1,7,1), 'lat':sst.lat.values, 'lon':sst.lon.values})
    
    # Construct mean, trend, and seasonal cycle
    mean = model_da[:,0].dot(sst_mod[0,:,:])
    trend = model_da[:,1].dot(sst_mod[1,:,:])
    seas = model_da[:,2:].dot(sst_mod[2:,:,:])
    
    # compute anomalies by removing all  the model coefficients 
    ssta_notrend = sst-model_da.dot(sst_mod)
    
    # Use the 90th percentile as a threshold and find anomalies that exceed it. 
    if ssta_notrend.chunks:
        ssta_notrend = ssta_notrend.chunk({'time': -1})
    
    threshold = ssta_notrend.quantile(.9, dim=('time'))
    features_notrend = ssta_notrend.where(ssta_notrend>=threshold, other=np.nan)
    return mean, trend, seas, features_notrend, ssta_notrend

## 1. Heat budget

### a) Load dataset

In [1]:
import xarray as xr

fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
target_vars = ['HDIFE_TEMP', 'HDIFN_TEMP', 'HDIFB_TEMP', 'KPP_SRC_TEMP']

for var in target_vars:
    fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
    import os
    print(f"{var}: {'exists' if os.path.exists(fpath + fname) else 'NOT FOUND'}")

HDIFE_TEMP: NOT FOUND
HDIFN_TEMP: NOT FOUND
HDIFB_TEMP: NOT FOUND
KPP_SRC_TEMP: NOT FOUND


In [6]:
#### FOSI
firstyear = 1979
lastyear = 2020

# Mask the Arctic, the Baltic Sea, the Red Sea, and the Black Sea
grid = pop_tools.get_grid('POP_gx1v7')
mask = xr.where((grid['REGION_MASK']>0) & (grid['REGION_MASK']<9), 1, np.nan)

field = 'TEMP'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi = xr.open_dataset(fpath+fname)

In [7]:
heat_budget_terms = xr.Dataset()

In [8]:
var = 'TEND_TEMP'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_TEND_TEMP = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_TEND_TEMP.isel(z_t = slice(0, 20))

In [8]:
var = 'TEND_TEMP'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_TEND_TEMP = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_TEND_TEMP.isel(z_t = slice(0, 30))

var = 'DIA_IMPVF_TEMP'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_DIA_IMPVF_TEMP = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_DIA_IMPVF_TEMP.isel(z_w_bot = slice(0, 30))

var = 'UET'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_UET = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_UET.isel(z_t = slice(0, 30))

var = 'VNT'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_VNT = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_VNT.isel(z_t = slice(0, 30))

var = 'WTT'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_WTT = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_WTT.isel(z_w_top = slice(0, 30))

var = 'TEMP'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_TEMP = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_TEMP.isel(z_t = slice(0, 30))

var = 'QSW_3D'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_QSW_3D = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_QSW_3D

var = 'SHF'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_SHF = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_SHF

var = 'SHF_QSW'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
ds_smyle_fosi_SHF_QSW = xr.open_dataset(fpath+fname)[var]
heat_budget_terms[var] = ds_smyle_fosi_SHF_QSW

# var = 'WVEL'
# fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
# ds_smyle_fosi_WVEL = xr.open_dataset(fpath+fname)[var]
# heat_budget_terms[var] = ds_smyle_fosi_WVEL.isel(z_w_top = slice(0, 30))

# var = 'UVEL'
# fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
# ds_smyle_fosi_UVEL = xr.open_dataset(fpath+fname)[var]
# heat_budget_terms[var] = ds_smyle_fosi_UVEL.isel(z_t = slice(0, 30))

# var = 'VVEL'
# fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{var}.030601-036812.nc'
# ds_smyle_fosi_VVEL = xr.open_dataset(fpath+fname)[var]
# heat_budget_terms[var] = ds_smyle_fosi_VVEL.isel(z_t = slice(0, 30))


In [9]:
heat_budget_terms['hflux_factor'] = ds_smyle_fosi['hflux_factor'].compute().item()

### b) Grid

In [10]:
%%time
# get lola inds from somewhere for indexing later on
lola_inds = {}
inds_lat = range(-89, 90, 1)
for j in inds_lat:
    if j < 0:
        lola_inds["j_" + str(j)[1:] + "s"] = pop_find_lat_ind(j, heat_budget_terms.TLAT)
    else:
        lola_inds["j_" + str(j) + "n"] = pop_find_lat_ind(j, heat_budget_terms.TLAT)

inds_lon = range(0, 360, 1)
for i in inds_lon:
    lola_inds["i_" + str(i) + "_w"] = pop_find_lon_ind(i, heat_budget_terms.TLONG)

CPU times: user 147 ms, sys: 3.63 ms, total: 151 ms
Wall time: 226 ms


In [11]:
%%time
DZT = np.zeros((30,384,320))

for k in range(30):
    DZT[k,:,:] = heat_budget_terms.z_t[k]

ds2 = xr.Dataset({'DZT': (['z_t','nlat','nlon',], DZT),
                   'DZU': (['z_t','nlat','nlon'], DZT)},
                    coords={'TLAT': (['nlat','nlon'],heat_budget_terms.TLAT.data),
                            'TLONG': (['nlat','nlon'],heat_budget_terms.TLONG.data),
                            'ULAT': (['nlat','nlon'],heat_budget_terms.ULAT.data),
                            'ULONG': (['nlat','nlon'],heat_budget_terms.ULONG.data),
                            'z_t': (['z_t'],heat_budget_terms.z_t.data)})

CPU times: user 7.91 ms, sys: 7.28 ms, total: 15.2 ms
Wall time: 97.3 ms


In [12]:
filepath_g = pop_tools.DATASETS.fetch('daily_surface_potential_temperature.nc')
ds_g = xr.open_dataset(filepath_g).isel(z_t = slice(0,30))

In [13]:
heat_budget_terms = heat_budget_terms.assign_coords(z_w=ds_g.z_w[:30])
heat_budget_terms = heat_budget_terms.assign_coords(z_w_bot=ds_g.z_w_bot[:30])

#### Set up vertical thickness and volume for scaling

In [14]:
fosi_montime_vals = [cftime.DatetimeNoLeap(1958+year, 1+month, 15) for year in range(63) for month in range(12)]
heat_budget_terms['time'] = fosi_montime_vals
heat_budget_terms = heat_budget_terms.sel(time=slice('1979-01', '2020-12'))#.isel(z_t = slice(0, 30))

In [15]:
%%time
heat_budget_terms["DXU"] = ds_g.DXU
heat_budget_terms["DYU"] = ds_g.DYU
heat_budget_terms["DXT"] = ds_g.DXT
heat_budget_terms["DYT"] = ds_g.DYT
heat_budget_terms["UAREA"] = ds_g.UAREA
heat_budget_terms["TAREA"] = ds_g.TAREA
heat_budget_terms["DZT"] = ds2.DZT
heat_budget_terms["DZU"] = ds2.DZU
heat_budget_terms["dz"] = heat_budget_terms.z_t

heat_budget_terms.DZT.attrs["long_name"] = "Thickness of T cells"
heat_budget_terms.DZT.attrs["units"] = "centimeter"
heat_budget_terms.DZT.attrs["grid_loc"] = "3111"
heat_budget_terms.DZU.attrs["long_name"] = "Thickness of U cells"
heat_budget_terms.DZU.attrs["units"] = "centimeter"
heat_budget_terms.DZU.attrs["grid_loc"] = "3221"

# make sure we have the cell volumne for calculations
VOL = (heat_budget_terms.DZT * heat_budget_terms.DXT * heat_budget_terms.DYT).compute()
KMT = ds_g.KMT.compute()

for j in tqdm(range(len(KMT.nlat))):
    for i in range(len(KMT.nlon)):
        k = KMT.values[j, i].astype(int)
        VOL.values[k:, j, i] = 0.0

heat_budget_terms["VOL"] = VOL

heat_budget_terms.VOL.attrs["long_name"] = "volume of T cells"
heat_budget_terms.VOL.attrs["units"] = "centimeter^3"
heat_budget_terms.VOL.attrs["grid_loc"] = "3111"

100%|██████████| 384/384 [00:00<00:00, 1648.79it/s]

CPU times: user 232 ms, sys: 6.73 ms, total: 239 ms
Wall time: 273 ms


In [16]:
budget = xr.Dataset()

In [ ]:
%%time
metrics = {
    ("X",): ["DXU", "DXT"],  # X distances
    ("Y",): ["DYU", "DYT"],  # Y distances
    ("Z",): ["DZU", "DZT"],  # Z distances
    ("X", "Y"): ["UAREA", "TAREA"],
}

gridxgcm, dsxgcm = pop_tools.to_xgcm_grid_dataset(
    heat_budget_terms,
    periodic=False,
    metrics=metrics,
    boundary={"X": "extend", "Y": "extend", "Z": "extend"},
)
  
for coord in ["nlat", "nlon"]:
    if coord in dsxgcm.coords:
        dsxgcm = dsxgcm.drop_vars(coord)

### c) Heat budget terms

#### 0) Tendency

In [ ]:
budget['TEND_TEMP'] = dsxgcm.TEND_TEMP

#### 1) Total heat advection

In [ ]:
budget_intermed = xr.Dataset()

In [ ]:
%%time

budget_intermed["UET"] = -(gridxgcm.diff(dsxgcm.UET * dsxgcm.VOL.values, axis="X") / dsxgcm.VOL)
budget_intermed["VNT"] = -(gridxgcm.diff(dsxgcm.VNT * dsxgcm.VOL.values, axis="Y") / dsxgcm.VOL)
budget_intermed["WTT"] = (
    gridxgcm.diff(dsxgcm.WTT.fillna(0) * (dsxgcm.dz * dsxgcm.DXT * dsxgcm.DYT).values, axis="Z")
    / dsxgcm.VOL
)
budget["TOT_ADV"] = budget_intermed["UET"] + budget_intermed["VNT"] + budget_intermed["WTT"]

#### 2) Heat advection due to vertical mixing

Total advection
Net surface heat flux
Shortwave radiative
Heat flux due to vertical mixing (DIA IMPVF TEMP and KPP)
Heat flux due to horizontal diffusion

Residual (subgrid): KPP from vertical mixing + Heat flux due to horizontal diffusion

In [ ]:
budget["DIA_IMPVF_TEMP"] = -(gridxgcm.diff(dsxgcm.DIA_IMPVF_TEMP * dsxgcm.TAREA, axis="Z") / dsxgcm.VOL)

In [ ]:
SRF_TEMP_FLUX = (dsxgcm.SHF) * dsxgcm.hflux_factor # do not remove SRW

In [ ]:
%%time
# budget["DIA_IMPVF_TEMP"] = -(gridxgcm.diff(dsxgcm.DIA_IMPVF_TEMP * dsxgcm.TAREA, axis="Z") / dsxgcm.VOL)

# SRF_TEMP_FLUX = (dsxgcm.SHF) * dsxgcm.hflux_factor # do not remove SRW

SRF_TEMP_FLUX = SRF_TEMP_FLUX.load()
first_term = (SRF_TEMP_FLUX*dsxgcm.TAREA)/dsxgcm.VOL.values[0, :, :]
# budget['SHF'] = first_term

# DIA_IMPVF_TEMP_ = dsxgcm.DIA_IMPVF_TEMP.isel(z_w_bot=0).load()
# second_term = DIA_IMPVF_TEMP_*dsxgcm.TAREA
# new_first_layer = (first_term + second_term)/dsxgcm.VOL.values[0, :, :]

# budget["DIA_IMPVF_TEMP"][:,0,:,:] = new_first_layer
#budget["VDIF_DIA_IMPVF_TEMP"] = budget["DIA_IMPVF_TEMP"] # missing KPP_SRC_TMP

In [ ]:
budget["SHF_depth"] = budget["DIA_IMPVF_TEMP"].copy()

In [ ]:
%%time
budget["SHF_depth"][:] = 0

In [ ]:
%%time
budget["SHF_depth"][:,0,:,:] = first_term

#### 3) Solar penetration

In [ ]:
%%time
budget["QSW_3D"] = -gridxgcm.diff((dsxgcm.QSW_3D * dsxgcm.hflux_factor), axis="Z") / dsxgcm.DZT